# Modelado de Keccak Modificado mediante MILP

Notebook limpio con el código relevante para reproducir los resultados descritos en el
informe: modelo MILP (DDT exacta y variables AND), verificación algebraica GF(2), búsqueda
constructiva de cotas superiores, y la grilla completa de experimentos (z = 4, 8; 1-3 rondas;
y la grilla ligada a los rangos de `intentos_fallidos` < 10, < 20, < 30).

Se omite el scaffolding genérico de AES en SageMath/Colab del notebook original (no se usa
para ningún resultado reportado); este notebook contiene únicamente la implementación en
Python + PuLP.


## Modelo específico para `keccak_f1600_variante` (Python + PuLP)

Este notebook implementa, en **Python puro + [PuLP](https://github.com/coin-or/pulp)**, el
modelo MILP usado para medir la resistencia diferencial de la variante de Keccak-f definida en
`implementacion.ipynb` (theta/rho/pi/chi/sigma/iota controlados por `intentos_fallidos`). Se
usa PuLP en vez de SageMath/Colab porque:

- No requiere Colab/condacolab/Sage: corre en cualquier entorno con `pip install pulp`.
- PuLP trae el solver CBC empaquetado y puede usar Gurobi u otros solvers si están
  disponibles, sin fallar si no lo están (requisito explícito del proyecto).

El estado se escala a un ancho de palabra `z` en vez de 64 bits, para que el MILP sea
tratable: `z=4` es `Keccak-f[100]` y `z=8` es `Keccak-f[200]`, las instancias oficiales más
pequeñas de la familia Keccak-f (mismas tablas `RC`/`ROTATION_OFFSETS`, tomadas módulo `z`).


In [1]:
!pip install -q pulp highspy --break-system-packages || pip install -q pulp highspy

In [6]:
import pulp
print("PuLP", pulp.__version__, "- solvers disponibles:", pulp.listSolvers(onlyAvailable=True))

PuLP 3.3.2 - solvers disponibles: ['PULP_CBC_CMD', 'HiGHS']


In [2]:
# Reutiliza RC y ROTATION_OFFSETS de implementacion.ipynb (no se redefinen, solo se escalan a z bits)
RC64 = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008,
]

ROTATION_OFFSETS = [
    [0, 36, 3, 41, 18],
    [1, 44, 10, 45, 2],
    [62, 6, 43, 15, 61],
    [28, 55, 25, 21, 56],
    [27, 20, 39, 8, 14],
]


def rotl(x, n, z):
    """ROTL64 de implementacion.ipynb, parametrizado por el ancho de palabra z."""
    n %= z
    mask = (1 << z) - 1
    x &= mask
    return x if n == 0 else ((x << n) & mask) | (x >> (z - n))


def theta(state, z):
    C = [0] * 5
    for x in range(5):
        v = 0
        for y in range(5):
            v ^= state[x][y]
        C[x] = v
    D = [C[(x - 1) % 5] ^ rotl(C[(x + 1) % 5], 1, z) for x in range(5)]
    for x in range(5):
        for y in range(5):
            state[x][y] ^= D[x]
    return state


def rho(state, z):
    for x in range(5):
        for y in range(5):
            state[x][y] = rotl(state[x][y], ROTATION_OFFSETS[x][y], z)
    return state


def pi_step(state, z):
    next_state = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            next_state[y][(2 * x + 3 * y) % 5] = state[x][y]
    return next_state


def chi(state, z):
    next_state = [[0] * 5 for _ in range(5)]
    mask = (1 << z) - 1
    for x in range(5):
        for y in range(5):
            next_state[x][y] = state[x][y] ^ ((~state[(x + 1) % 5][y]) & state[(x + 2) % 5][y] & mask)
    return next_state


def sigma(state, intentos_fallidos, z):
    mask = (1 << z) - 1
    for x in range(5):
        for y in range(5):
            state[x][y] ^= (intentos_fallidos << (x + y)) & mask
            state[x][y] = rotl(state[x][y], (intentos_fallidos + x + y) % z, z)
    return state


def iota_step(state, round_idx, intentos_fallidos, z):
    rc_mod = (RC64[round_idx] & ((1 << z) - 1)) ^ (intentos_fallidos & ((1 << z) - 1))
    state[0][0] ^= rc_mod
    return state


def keccak_f_variante(state, intentos_fallidos, z, num_rounds=None):
    """keccak_f1600_variante de implementacion.ipynb, escalado a z bits de palabra."""
    rondas = num_rounds if num_rounds is not None else min(24, 8 + intentos_fallidos)
    for round_idx in range(rondas):
        state = theta(state, z)
        state = rho(state, z)
        if intentos_fallidos < 3:
            state = pi_step(state, z)
            state = chi(state, z)
        else:
            state = chi(state, z)
            state = pi_step(state, z)
            state = sigma(state, intentos_fallidos, z)
        state = iota_step(state, round_idx, intentos_fallidos, z)
    return state


print("Referencia funcional lista (rondas(intentos_fallidos) = min(24, 8+intentos_fallidos)).")
print("rondas(0)=", min(24, 8), " rondas(9)=", min(24, 17), " rondas(19)=", min(24, 27), " rondas(29)=", min(24, 37))


Referencia funcional lista (rondas(intentos_fallidos) = min(24, 8+intentos_fallidos)).
rondas(0)= 8  rondas(9)= 17  rondas(19)= 24  rondas(29)= 24


### Qué afecta realmente a una diferencial y qué no

Un trail diferencial sigue cómo se propaga Δ = a ⊕ b para dos entradas relacionadas. Esto
importa para modelar el MILP: **si un paso hace XOR a ambos lados de un par diferencial con la
misma constante conocida (que no depende de los datos), la diferencia Δ no cambia**, porque
`(a⊕c) ⊕ (b⊕c) = a⊕b`. De las cinco alteraciones de la variante, esto aplica directamente a
dos de ellas:

- **`iota`**: el XOR de `state[0][0]` con `RC[i] ^ intentos_fallidos` es una constante fija
  para una ronda e `intentos_fallidos` dados, igual para ambos lados del par diferencial.
  **No afecta la propagación diferencial**, exactamente igual que en el Keccak original (RC
  tampoco lo hace ahí).
- **`sigma`**: el término `state[x][y] ^= (intentos_fallidos << (x+y))` es la misma historia:
  constante conocida, sin efecto en Δ. Solo la **rotación** que sigue
  (`ROTL64(state[x][y], (intentos_fallidos+x+y) % z)`) sí actúa sobre la diferencia (rota Δ,
  igual que `rho`).

Esto es relevante para el análisis de seguridad de la propuesta (ver conclusión): la
afirmación del README de que la parte dinámica de `sigma`/`iota` "dificultaría ataques que
necesitan del determinismo" **no se sostiene frente a criptoanálisis diferencial clásico**: el término XOR-constante es transparente a Δ. Lo único que sí cambia la resistencia
diferencial es (a) el número de rondas efectivas y (b) el reordenamiento chi↔pi + la rotación
de `sigma`.

En consecuencia, el modelo MILP de abajo **no necesita variables para `sigma` (la parte XOR)
ni para `iota`**: solo theta, rho, la posible rotación de `sigma`, pi y chi (la única no
lineal) entran al modelo de propagación de diferencias.


In [3]:
# Tabla de distribución de diferencias (DDT) exacta de la fila de 5 bits de chi,
# obtenida por fuerza bruta (2^5 x 2^5 = 1024 casos) en vez de asumir una fórmula de memoria:
# el peso de una fila activa depende del patrón exacto de bits, no solo de su peso de Hamming
# (ej. 00111 -> peso 4, pero 01011 -> peso 3, con el mismo número de bits activos).
import math
from collections import defaultdict


def chi_row(a):
    return tuple(a[x] ^ ((1 - a[(x + 1) % 5]) & a[(x + 2) % 5]) for x in range(5))


def _to_bits(v):
    return tuple((v >> i) & 1 for i in range(5))


def _to_int(bits):
    return sum(b << i for i, b in enumerate(bits))


def build_chi_transitions():
    ddt = defaultdict(lambda: defaultdict(int))
    for a in range(32):
        ain = _to_bits(a)
        outa = chi_row(ain)
        for d in range(32):
            din = _to_bits(d)
            bin_ = tuple(ain[i] ^ din[i] for i in range(5))
            outb = chi_row(bin_)
            diff_out = tuple(outa[i] ^ outb[i] for i in range(5))
            ddt[d][_to_int(diff_out)] += 1
    transitions = []
    for d in range(32):
        din = _to_bits(d)
        for out_val, cnt in ddt[d].items():
            transitions.append((din, _to_bits(out_val), -math.log2(cnt / 32)))
    return transitions


CHI_TRANSITIONS = build_chi_transitions()
nonzero = [t for t in CHI_TRANSITIONS if any(t[0])]
print("Transiciones válidas (entrada no nula):", len(nonzero))
print("Pesos distintos observados:", sorted(set(t[2] for t in nonzero)))
print("-> el peso mínimo posible de una fila activa es 2 (ninguna transición de peso 1 existe)")


Transiciones válidas (entrada no nula): 316
Pesos distintos observados: [2.0, 3.0, 4.0]
-> el peso mínimo posible de una fila activa es 2 (ninguna transición de peso 1 existe)


### Modelo MILP: variables binarias por bit, chi linealizado exacto

Cada bit de diferencia del estado, en cada ronda, es una variable binaria. `theta`, `rho`,
`pi` y la rotación de `sigma` son GF(2)-lineales o simples permutaciones de bits, así que se
aplican **sin variables nuevas** cuando son solo una permutación (rho/pi/sigma-rotación son
un reindexado de variables ya existentes) y con la linealización XOR estándar
(`c <= a+b; a <= b+c; b <= a+c; a+b+c <= 2`) cuando combinan bits (theta).

`chi` es la única capa no lineal y usa un **selector one-hot exacto** sobre las 317
transiciones válidas de la DDT calculada arriba (316 con entrada no nula + la transición
trivial 0→0): por cada fila (`y`, bit `b`, con las 5 lanas `x=0..4`) se crea una variable
binaria por transición válida, se fuerza que exactamente una esté activa, se atan los bits de
entrada/salida de esa fila a la transición elegida, y su peso entra a la función objetivo.
Esto es exacto (no una relajación heurística tipo "branch number"), a diferencia de un modelo
tipo "branch number" que no distingue transiciones dentro de un mismo peso de Hamming.


In [4]:
import time


def _new_binary(name):
    return pulp.LpVariable(name, cat="Binary")


def _xor2(prob, a, b, name):
    c = pulp.LpVariable(name, cat="Binary")
    prob += c <= a + b
    prob += a <= b + c
    prob += b <= a + c
    prob += a + b + c <= 2
    return c


def _xor_n(prob, vars_, name_prefix):
    cur = vars_[0]
    for i in range(1, len(vars_)):
        cur = _xor2(prob, cur, vars_[i], f"{name_prefix}_t{i}")
    return cur


def rotl_bits(bit_array, n, z):
    n %= z
    return [bit_array[(i - n) % z] for i in range(z)]


def make_lane_grid(z, name_prefix):
    return [[[_new_binary(f"{name_prefix}_x{x}y{y}b{b}") for b in range(z)] for y in range(5)] for x in range(5)]


def apply_theta_milp(prob, state, z, round_idx):
    C = []
    for x in range(5):
        col_bits = []
        for b in range(z):
            bits = [state[x][y][b] for y in range(5)]
            col_bits.append(_xor_n(prob, bits, f"r{round_idx}_C_x{x}_b{b}"))
        C.append(col_bits)
    D = []
    for x in range(5):
        left = C[(x - 1) % 5]
        right_rot = rotl_bits(C[(x + 1) % 5], 1, z)
        D.append([_xor2(prob, left[b], right_rot[b], f"r{round_idx}_D_x{x}_b{b}") for b in range(z)])
    theta_out = [[None] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            theta_out[x][y] = [_xor2(prob, state[x][y][b], D[x][b], f"r{round_idx}_theta_x{x}y{y}b{b}") for b in range(z)]
    return theta_out


def apply_rho_milp(state, z):
    return [[rotl_bits(state[x][y], ROTATION_OFFSETS[x][y], z) for y in range(5)] for x in range(5)]


def apply_pi_milp(state):
    out = [[None] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            out[y][(2 * x + 3 * y) % 5] = state[x][y]
    return out


def apply_sigma_rotation_milp(state, z, intentos_fallidos):
    out = [[None] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            shift = (intentos_fallidos + x + y) % z
            out[x][y] = rotl_bits(state[x][y], shift, z)
    return out


def apply_chi_milp(prob, state, z, round_idx, objective_terms, active_indicators):
    out = [[[None] * z for _ in range(5)] for _ in range(5)]
    for y in range(5):
        for b in range(z):
            row_in = [state[x][y][b] for x in range(5)]
            row_out = [pulp.LpVariable(f"r{round_idx}_chiout_y{y}b{b}_x{x}", cat="Binary") for x in range(5)]
            t_vars = []
            for k, (din, dout, w) in enumerate(CHI_TRANSITIONS):
                t = pulp.LpVariable(f"r{round_idx}_sel_y{y}b{b}_k{k}", cat="Binary")
                t_vars.append((t, din, dout, w))
            prob += pulp.lpSum(t for t, _, _, _ in t_vars) == 1
            for i in range(5):
                prob += row_in[i] == pulp.lpSum(t for t, din, _, _ in t_vars if din[i] == 1)
                prob += row_out[i] == pulp.lpSum(t for t, _, dout, _ in t_vars if dout[i] == 1)
            objective_terms.extend((t, w) for t, _, _, w in t_vars if w > 0)
            active_indicators.append((f"r{round_idx}_y{y}_b{b}", [t for t, din, _, _ in t_vars if any(din)]))
            for x in range(5):
                out[x][y][b] = row_out[x]
    return out


def _parse_solver_log(log_text):
    """Devuelve (proven_optimal, lower_bound) leyendo el log nativo de CBC o HiGHS -- pulp
    a veces reporta 'Optimal' aunque el solver solo llego al limite de tiempo."""
    proven = None
    if "Stopped on time limit" in log_text or "Time limit reached" in log_text:
        proven = False
    elif "Optimal solution found" in log_text or "Status            Optimal" in log_text:
        proven = True
    lower_bound = None
    for line in log_text.splitlines():
        line = line.strip()
        if line.startswith("Lower bound:"):
            try:
                lower_bound = float(line.split(":", 1)[1].strip())
            except ValueError:
                pass
        if line.startswith("Dual bound"):
            try:
                lower_bound = float(line.split()[-1])
            except ValueError:
                pass
    return proven, lower_bound


def pick_solver(time_limit, msg, log_path):
    try:
        if pulp.GUROBI_CMD().available():
            return pulp.GUROBI_CMD(msg=msg, timeLimit=time_limit)
    except Exception:
        pass
    if "HiGHS" in pulp.listSolvers(onlyAvailable=True):
        return pulp.HiGHS(msg=msg, timeLimit=time_limit)
    return pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit, logPath=log_path)


def build_trail_model(z, num_rounds, intentos_fallidos, time_limit=120, msg=False):
    prob = pulp.LpProblem("keccak_variant_trail", pulp.LpMinimize)
    state = make_lane_grid(z, "s0")
    all_bits0 = [state[x][y][b] for x in range(5) for y in range(5) for b in range(z)]
    prob += pulp.lpSum(all_bits0) >= 1  # excluye el trail trivial (todo ceros)

    objective_terms, active_indicators = [], []
    for r in range(num_rounds):
        theta_out = apply_theta_milp(prob, state, z, r)
        rho_out = apply_rho_milp(theta_out, z)
        if intentos_fallidos < 3:
            pi_out = apply_pi_milp(rho_out)
            state = apply_chi_milp(prob, pi_out, z, r, objective_terms, active_indicators)
        else:
            chi_out = apply_chi_milp(prob, rho_out, z, r, objective_terms, active_indicators)
            pi_out = apply_pi_milp(chi_out)
            state = apply_sigma_rotation_milp(pi_out, z, intentos_fallidos)

    prob += pulp.lpSum(t * w for t, w in objective_terms)

    log_path = f"/tmp/solver_log_{id(prob)}.txt"
    solver = pick_solver(time_limit, msg, log_path)
    t0 = time.time()
    prob.solve(solver)
    elapsed = time.time() - t0

    raw_status = pulp.LpStatus[prob.status]
    obj_val = pulp.value(prob.objective)
    try:
        log_text = open(log_path).read()
    except FileNotFoundError:
        log_text = ""
    proven_optimal, lower_bound = _parse_solver_log(log_text)

    n_active = sum(1 for _, ts in active_indicators if any((pulp.value(t) or 0) > 0.5 for t in ts))

    if raw_status != "Optimal":
        status = raw_status
    elif proven_optimal is True:
        status = "Optimal (proven)"
    elif proven_optimal is False:
        status = "TimeLimit (best found, NOT proven optimal)"
    else:
        status = raw_status

    return {
        "z": z, "rounds": num_rounds, "intentos_fallidos": intentos_fallidos,
        "status": status, "weight": obj_val, "lower_bound": lower_bound,
        "active_sboxes": n_active, "solve_time_s": round(elapsed, 2),
        "solver": getattr(solver, "name", str(solver)),
    }


print("Modelo MILP definido.")


Modelo MILP definido.


In [7]:
# 1 ronda: el MILP exacto SI converge y prueba optimalidad, para z=4 y z=8, en ambos ordenes
# (intentos_fallidos=0 -> orden clasico; intentos_fallidos=5 -> orden con chi/pi intercambiados + sigma)
round1_results = []
for z in (4, 8):
    for intentos in (0, 5):
        res = build_trail_model(z=z, num_rounds=1, intentos_fallidos=intentos, time_limit=120, msg=False)
        round1_results.append(res)
        print(res)


{'z': 4, 'rounds': 1, 'intentos_fallidos': 0, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'active_sboxes': 1, 'solve_time_s': 8.3, 'solver': 'HiGHS'}
{'z': 4, 'rounds': 1, 'intentos_fallidos': 5, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'active_sboxes': 1, 'solve_time_s': 8.75, 'solver': 'HiGHS'}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 0, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'active_sboxes': 1, 'solve_time_s': 22.12, 'solver': 'HiGHS'}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 5, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'active_sboxes': 1, 'solve_time_s': 16.0, 'solver': 'HiGHS'}


### Cumplimiento literal de la tarea (b): linealización de chi con variables AND

El enunciado original (tarea específica **b**) pide explícitamente linealizar la operación no
lineal χ "usando variables auxiliares AND", en vez del selector one-hot sobre la DDT completa
usado arriba. Ambas son formas válidas de modelar χ en MILP, pero no son equivalentes, así que
se implementan **las dos** en este notebook: la DDT/one-hot (ya definida, exacta, usada para
los resultados de la sección 5) y, a continuación, la versión literal pedida por el enunciado,
para verificar si también es utilizable y comparar su precisión.

χ, fila por fila, es `chi_row(a)[x] = a[x] ^ (NOT(a[x+1]) & a[x+2])`. El `NOT` es afín (XOR con
la constante 1), así que no cambia la propagación de una diferencia: `d(NOT a) = d(a)`. Por
tanto el término AND se modela directamente sobre las variables de diferencia `d[x+1]`,
`d[x+2]` de la fila (sin necesitar una variable para el NOT). Para un AND de dos bits de
diferencia `(da, db)`, la salida real depende de los valores concretos de los bits (no solo de
la diferencia):

- Si `da = db = 0`, la diferencia de salida es forzosamente `0` (peso 0, determinista).
- Si al menos una de las dos está activa, la diferencia de salida puede ser `0` o `1` con igual
  probabilidad (depende del valor concreto, no conocido de antemano); consume **1 bit de
  peso**, sin importar si una o las dos entradas del AND están activas.

Esto se modela con una variable `w` (indicador de peso, `w = OR(da, db)`) y una variable `t`
(la diferencia de salida del término AND, libre cuando `w=1`, forzada a 0 si `w=0`):

```
w >= da;  w >= db;  w <= da + db      # w = OR(da, db)
t <= w                                  # t libre si w=1, forzado a 0 si w=0
salida_x = a[x] XOR t                  # XOR estandar (mismas 4 restricciones que theta)
```

y se suma `w` (peso 1 por cada AND activo) a la función objetivo.


In [8]:
def apply_chi_milp_and_gates(prob, state, z, round_idx, objective_terms):
    """Linealizacion literal de chi via variables auxiliares AND (tarea (b) del enunciado).
    Reutiliza _xor2 y pulp ya definidos arriba. Por cada fila (y,b) y cada posicion x se crea
    un gate AND entre las diferencias de entrada rotadas (x+1, x+2): w = OR(da,db) entra al
    objetivo con peso 1, t <= w es la diferencia de salida del termino AND (libre si w=1)."""
    out = [[[None] * z for _ in range(5)] for _ in range(5)]
    for y in range(5):
        for b in range(z):
            row_in = [state[x][y][b] for x in range(5)]
            for x in range(5):
                da = row_in[(x + 1) % 5]
                db = row_in[(x + 2) % 5]
                w = pulp.LpVariable(f"r{round_idx}_andw_y{y}b{b}_x{x}", cat="Binary")
                prob += w >= da
                prob += w >= db
                prob += w <= da + db
                t = pulp.LpVariable(f"r{round_idx}_andt_y{y}b{b}_x{x}", cat="Binary")
                prob += t <= w
                objective_terms.append((w, 1))
                out[x][y][b] = _xor2(prob, row_in[x], t, f"r{round_idx}_chiout_and_y{y}b{b}_x{x}")
    return out


def build_trail_model_and_gates(z, num_rounds, intentos_fallidos, time_limit=120, msg=False):
    """Igual que build_trail_model, pero linealizando chi con variables AND en vez del
    selector one-hot sobre la DDT (cumplimiento literal de la tarea (b))."""
    prob = pulp.LpProblem("keccak_variant_trail_and", pulp.LpMinimize)
    state = make_lane_grid(z, "s0and")
    all_bits0 = [state[x][y][b] for x in range(5) for y in range(5) for b in range(z)]
    prob += pulp.lpSum(all_bits0) >= 1

    objective_terms = []
    for r in range(num_rounds):
        theta_out = apply_theta_milp(prob, state, z, r)
        rho_out = apply_rho_milp(theta_out, z)
        if intentos_fallidos < 3:
            pi_out = apply_pi_milp(rho_out)
            state = apply_chi_milp_and_gates(prob, pi_out, z, r, objective_terms)
        else:
            chi_out = apply_chi_milp_and_gates(prob, rho_out, z, r, objective_terms)
            pi_out = apply_pi_milp(chi_out)
            state = apply_sigma_rotation_milp(pi_out, z, intentos_fallidos)

    prob += pulp.lpSum(t * w for t, w in objective_terms)

    log_path = f"/tmp/solver_log_and_{id(prob)}.txt"
    solver = pick_solver(time_limit, msg, log_path)
    t0 = time.time()
    prob.solve(solver)
    elapsed = time.time() - t0

    raw_status = pulp.LpStatus[prob.status]
    obj_val = pulp.value(prob.objective)
    try:
        log_text = open(log_path).read()
    except FileNotFoundError:
        log_text = ""
    proven_optimal, lower_bound = _parse_solver_log(log_text)

    if raw_status != "Optimal":
        status = raw_status
    elif proven_optimal is True:
        status = "Optimal (proven)"
    elif proven_optimal is False:
        status = "TimeLimit (best found, NOT proven optimal)"
    else:
        status = raw_status

    return {
        "z": z, "rounds": num_rounds, "intentos_fallidos": intentos_fallidos,
        "status": status, "weight": obj_val, "lower_bound": lower_bound,
        "solve_time_s": round(elapsed, 2), "solver": getattr(solver, "name", str(solver)),
    }


# Mismos 4 casos de 1 ronda que la version DDT exacta (cell round1_results), para comparar
and_gate_results = []
for z in (4, 8):
    for intentos in (0, 5):
        res = build_trail_model_and_gates(z=z, num_rounds=1, intentos_fallidos=intentos, time_limit=120, msg=False)
        and_gate_results.append(res)
        print(res)


{'z': 4, 'rounds': 1, 'intentos_fallidos': 0, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'solve_time_s': 7.12, 'solver': 'HiGHS'}
{'z': 4, 'rounds': 1, 'intentos_fallidos': 5, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'solve_time_s': 9.31, 'solver': 'HiGHS'}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 0, 'status': 'Optimal', 'weight': 2.0, 'lower_bound': None, 'solve_time_s': 120.14, 'solver': 'HiGHS'}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 5, 'status': 'Optimal', 'weight': 2.000000000000003, 'lower_bound': None, 'solve_time_s': 37.56, 'solver': 'HiGHS'}


### Comparación: variables AND (literal) vs. DDT/one-hot (exacta)

La salida de la celda anterior muestra, para **esta ejecución concreta**, cuántos de los
cuatro casos de validación de 1 ronda alcanzan el óptimo probado (peso 2) con la versión AND.
Esto puede variar entre corridas: la convergencia y el tiempo de resolución de un MILP
genérico dependen de detalles no deterministas del branch-and-bound (orden de ramificación,
hardware, versión del solver). Por ejemplo, en la corrida documentada originalmente en el
informe, el caso `z=8` con el orden variante no lograba probar optimalidad dentro del límite
de tiempo de 120 s y reportaba un peso mayor sin probar (`peso 4`, `TimeLimit`, con el log
nativo mostrando `Stopped on time limit` / `Lower bound: 0.000`). Esta variabilidad en el
tiempo de cómputo **no afecta la conclusión estructural siguiente**, que es independiente de
si el solver converge o no en una corrida particular:

- **Por qué la versión AND no es equivalente a la DDT, incluso cuando converge al mismo
  valor.** El modelo AND linealiza cada uno de los 5 términos AND de una fila **de forma
  independiente** (peso 1 si al menos una de sus dos entradas está activa), sin imponer que
  la combinación conjunta de las 5 salidas corresponda a una transición realmente alcanzable
  por `chi_row`. La DDT, en cambio, selecciona una transición completa y consistente de las
  316 observadas por fuerza bruta. Un ejemplo concreto con la propia DDT lo muestra: para el
  patrón de entrada `00111` (bits 2,3,4 activos), contar los AND activos por posición da 4,
  que coincide con el peso real (4). Pero para `01011` (bits 1,3,4 activos, mismo número de
  bits activos), contar los AND activos da **5**, mientras que el peso real medido por la DDT
  es **3**. El modelo AND sobreestima en este caso, porque no ve que dos de esos 5 "gates
  activos" comparten condiciones y no consumen peso independiente. Esta discrepancia es
  estructural (se puede verificar contando a mano sobre la DDT ya calculada) y no depende de
  qué tan rápido converja el solver en una corrida dada.
- **Consecuencia práctica.** La versión AND es una aproximación válida para verificar el
  requisito literal del enunciado, pero **no es la que se usa para los resultados de la
  sección de resultados**: esos usan la DDT/one-hot exacta (secciones anteriores), que es
  estrictamente más precisa por construcción y ya fue verificada por un segundo método
  independiente (inversión GF(2) de `theta∘rho`), que no depende del solver en absoluto ni de
  su tiempo de ejecución. Se deja la versión AND implementada y ejecutada como evidencia de
  que ambos enfoques fueron evaluados, documentando explícitamente por qué se prefirió la
  alternativa exacta para el resto del análisis.


### Límite práctico del MILP genérico más allá de 1 ronda

El modelo de arriba es **exacto** (usa la DDT real de chi, no una relajación tipo "branch
number"), pero su relajación LP resulta muy débil: tanto CBC como
[HiGHS](https://highs.dev/) pueden llegar al límite de tiempo con **cota inferior (dual
bound) estancada en 0** para 2 o más rondas, y en ocasiones incluso para 1 ronda con `z=8`
(dependiendo de la corrida, como se discutió arriba para el modelo AND). Esto es una
limitación conocida de las codificaciones "one-hot"/AND de S-boxes en MILP genérico (sin las
desigualdades reducidas por envolvente convexa que usan los trabajos de criptoanálisis
dedicados). **No es un error del modelo**, que sigue siendo correcto; solo el solver
genérico no logra demostrar optimalidad de forma confiable a esta escala en un tiempo
razonable.

**Alternativa que sí resuelve la ronda 1 sin depender del solver:** `theta` y `rho` son
biyecciones lineales invertibles sobre GF(2). Por lo tanto, para *cualquier* patrón disperso
deseado a la entrada de chi (p. ej. un único bit activo, que activa exactamente 1 fila de
chi), existe un único estado previo a la ronda que lo produce, obtenido invirtiendo
`theta∘rho`. Como el peso mínimo de una fila activa es 2 (ninguna transición de peso 1 existe
en la DDT), **el peso mínimo de un trail de 1 ronda es exactamente 2, para cualquier `z` y
cualquier orden de pasos**, un resultado demostrable algebraicamente, no solo hallado por
búsqueda ni dependiente de que el solver converja. Se usa esta construcción explícita (en vez
de esperar al MILP) para validar el caso de 1 ronda de forma robusta e independiente del
solver, y como punto de partida para una búsqueda constructiva (greedy) en rondas
posteriores, donde el MILP exacto ya no converge en un tiempo práctico.


In [9]:
# Inversion generica GF(2) de theta+rho (algebra lineal pura, sin heuristicas de por medio):
# se construye la matriz aplicando la funcion a cada vector base, y se invierte por Gauss-Jordan.

def state_to_flat(state, z):
    return [(state[x][y] >> b) & 1 for x in range(5) for y in range(5) for b in range(z)]


def flat_to_state(bits, z):
    state = [[0] * 5 for _ in range(5)]
    idx = 0
    for x in range(5):
        for y in range(5):
            v = 0
            for b in range(z):
                v |= bits[idx] << b
                idx += 1
            state[x][y] = v
    return state


def build_matrix_columns(linear_fn, z):
    n = 25 * z
    columns = []
    for i in range(n):
        bits = [0] * n
        bits[i] = 1
        columns.append(state_to_flat(linear_fn(flat_to_state(bits, z), z), z))
    return columns


def invert_columns(columns, n):
    rows = []
    for r in range(n):
        m_bits = 0
        for c in range(n):
            if columns[c][r]:
                m_bits |= 1 << c
        rows.append(m_bits | (1 << (n + r)))
    for col in range(n):
        pivot = next(r for r in range(col, n) if (rows[r] >> col) & 1)
        rows[col], rows[pivot] = rows[pivot], rows[col]
        for r in range(n):
            if r != col and (rows[r] >> col) & 1:
                rows[r] ^= rows[col]
    inv_columns = [[0] * n for _ in range(n)]
    for r in range(n):
        inv_part = rows[r] >> n
        for c in range(n):
            inv_columns[c][r] = (inv_part >> c) & 1
    return inv_columns


def apply_matrix(columns, flat_bits):
    n = len(flat_bits)
    out = [0] * n
    for c in range(n):
        if flat_bits[c]:
            for r in range(n):
                out[r] ^= columns[c][r]
    return out


class LinearInverter:
    def __init__(self, linear_fn, z):
        self.z = z
        cols = build_matrix_columns(linear_fn, z)
        self.inv_columns = invert_columns(cols, 25 * z)

    def preimage(self, target_state):
        return flat_to_state(apply_matrix(self.inv_columns, state_to_flat(target_state, self.z)), self.z)


def theta_rho_diff(state, z):
    C = [0] * 5
    for x in range(5):
        v = 0
        for y in range(5):
            v ^= state[x][y]
        C[x] = v
    D = [C[(x - 1) % 5] ^ rotl(C[(x + 1) % 5], 1, z) for x in range(5)]
    theta_out = [[state[x][y] ^ D[x] for y in range(5)] for x in range(5)]
    return [[rotl(theta_out[x][y], ROTATION_OFFSETS[x][y], z) for y in range(5)] for x in range(5)]


# verificacion: invertir y volver a aplicar debe reproducir el estado original
import random

rng = random.Random(2026)
for z in (4, 8):
    inv = LinearInverter(theta_rho_diff, z)
    mask = (1 << z) - 1
    for _ in range(5):
        s = [[rng.getrandbits(z) & mask for _ in range(5)] for _ in range(5)]
        fwd = theta_rho_diff(s, z)
        assert inv.preimage(fwd) == s
    print(f"z={z}: inversion de theta-rho verificada sobre estados aleatorios")


z=4: inversion de theta-rho verificada sobre estados aleatorios
z=8: inversion de theta-rho verificada sobre estados aleatorios


In [10]:
# Construccion concreta (no-MILP) de un trail: propaga un patron de diferencia real ronda a
# ronda, usando la DDT exacta de chi para elegir, en cada fila activa, la transicion de MENOR
# peso disponible (empate roto por menor peso de Hamming en la salida, para no generar mas
# actividad de la necesaria). Esto da una cota superior VALIDA (un trail real, verificable),
# no necesariamente el minimo global -- pero el minimo global de 1 ronda ya esta probado arriba.
from collections import defaultdict

PATTERN_TRANSITIONS = defaultdict(list)
for din, dout, w in CHI_TRANSITIONS:
    d = sum(b << i for i, b in enumerate(din))
    o = sum(b << i for i, b in enumerate(dout))
    PATTERN_TRANSITIONS[d].append((o, w))
for d in PATTERN_TRANSITIONS:
    PATTERN_TRANSITIONS[d].sort(key=lambda ow: (ow[1], bin(ow[0]).count("1")))


def pi_diff(state):
    out = [[0] * 5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            out[y][(2 * x + 3 * y) % 5] = state[x][y]
    return out


def sigma_rot_diff(state, intentos_fallidos, z):
    return [[rotl(state[x][y], (intentos_fallidos + x + y) % z, z) for y in range(5)] for x in range(5)]


def chi_diff_greedy(state, z):
    out = [[0] * 5 for _ in range(5)]
    weight = 0
    active = 0
    for y in range(5):
        for b in range(z):
            d = 0
            for x in range(5):
                d |= ((state[x][y] >> b) & 1) << x
            if d == 0:
                continue
            active += 1
            out_pattern, w = PATTERN_TRANSITIONS[d][0]
            weight += w
            for x in range(5):
                out[x][y] |= ((out_pattern >> x) & 1) << b
    return out, weight, active


def run_trail(initial_state, z, num_rounds, intentos_fallidos):
    state = [row[:] for row in initial_state]
    total_weight = total_active = 0
    for _ in range(num_rounds):
        state = theta(state, z)
        state = rho(state, z)
        if intentos_fallidos < 3:
            state = pi_diff(state)
            state, w, a = chi_diff_greedy(state, z)
        else:
            state, w, a = chi_diff_greedy(state, z)
            state = pi_diff(state)
            state = sigma_rot_diff(state, intentos_fallidos, z)
        total_weight += w
        total_active += a
    return total_weight, total_active


def search_best_trail(z, num_rounds, intentos_fallidos, inverter):
    """Prueba, para cada una de las 25*z posiciones de un unico bit activo justo antes de chi
    en la ronda 0 (invirtiendo theta-rho), la propagacion resultante; se queda con la mejor."""
    best = None
    for x0 in range(5):
        for y0 in range(5):
            for b0 in range(z):
                target = [[0] * 5 for _ in range(5)]
                target[x0][y0] = 1 << b0
                state0 = inverter.preimage(target)
                w, a = run_trail(state0, z, num_rounds, intentos_fallidos)
                if best is None or w < best[0]:
                    best = (w, a)
    return best


print("Busqueda constructiva (greedy) definida.")


Busqueda constructiva (greedy) definida.


In [11]:
# Grilla de experimentos: z en {4,8} x rondas en {1,2,3}, ambos ordenes de pasos,
# mas los conteos de rondas asociados a los rangos de intentos_fallidos pedidos
# (rondas = min(24, 8+intentos_fallidos)): <10 -> hasta 17 rondas; <20 y <30 -> saturan en 24.
inverters = {z: LinearInverter(theta_rho_diff, z) for z in (4, 8)}

grid_small = [(z, r, i) for z in (4, 8) for r in (1, 2, 3) for i in (0, 5)]
grid_large = [(z, r, i) for z in (4, 8) for r, i in [(8, 0), (17, 9), (24, 19)]]

heuristic_results = []
for z, rounds, intentos in grid_small + grid_large:
    w, a = search_best_trail(z, rounds, intentos, inverters[z])
    heuristic_results.append({"z": z, "rounds": rounds, "intentos_fallidos": intentos, "weight_upper_bound": w, "active_sboxes": a})

for r in heuristic_results:
    print(r)


{'z': 4, 'rounds': 1, 'intentos_fallidos': 0, 'weight_upper_bound': 2.0, 'active_sboxes': 1}
{'z': 4, 'rounds': 1, 'intentos_fallidos': 5, 'weight_upper_bound': 2.0, 'active_sboxes': 1}
{'z': 4, 'rounds': 2, 'intentos_fallidos': 0, 'weight_upper_bound': 21.0, 'active_sboxes': 9}
{'z': 4, 'rounds': 2, 'intentos_fallidos': 5, 'weight_upper_bound': 20.0, 'active_sboxes': 8}
{'z': 4, 'rounds': 3, 'intentos_fallidos': 0, 'weight_upper_bound': 62.0, 'active_sboxes': 25}
{'z': 4, 'rounds': 3, 'intentos_fallidos': 5, 'weight_upper_bound': 70.0, 'active_sboxes': 30}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 0, 'weight_upper_bound': 2.0, 'active_sboxes': 1}
{'z': 8, 'rounds': 1, 'intentos_fallidos': 5, 'weight_upper_bound': 2.0, 'active_sboxes': 1}
{'z': 8, 'rounds': 2, 'intentos_fallidos': 0, 'weight_upper_bound': 21.0, 'active_sboxes': 9}
{'z': 8, 'rounds': 2, 'intentos_fallidos': 5, 'weight_upper_bound': 21.0, 'active_sboxes': 9}
{'z': 8, 'rounds': 3, 'intentos_fallidos': 0, 'weight_upper_bo

## Resultados y análisis de seguridad

**1 ronda (probado, no solo hallado por búsqueda):** el peso mínimo de un trail diferencial es
exactamente **2** (probabilidad 2⁻² = 1/4, ~4 pares), igual para `z=4` y `z=8`, y para el
orden clásico y el de la variante. Se prueba tanto por MILP exacto (converge y demuestra
optimalidad con HiGHS) como algebraicamente (inversión de `theta∘rho`, sección anterior). El
reordenamiento chi↔pi de la variante **no cambia el mínimo de 1 ronda**: `pi` solo reubica
lanas completas, así que una fila con un único bit activo tras `rho` sigue teniendo un único
bit activo tras `pi`, sin importar en qué punto del pipeline se inserte.

**2+ rondas (cota superior constructiva, no óptimo probado):** el MILP exacto no converge en
un tiempo práctico más allá de 1 ronda (ver limitación de la sección anterior), así que los
valores de la tabla para 2, 3, 8, 17 y 24 rondas son trails **reales y verificables**
(construidos invirtiendo `theta∘rho` y avanzando con la mejor transición de chi disponible en
cada fila activa), es decir, **cotas superiores** honestas sobre el peso mínimo verdadero: el
atacante no necesita más pares que los indicados, aunque podría necesitar menos. El
crecimiento observado es rápido: de peso 2 en 1 ronda pasa a un rango de docenas en 2-3
rondas, y a cientos incluso en el escenario **más débil posible** (`intentos_fallidos = 0`,
8 rondas, orden clásico); ya en ese punto el número de pares requerido (2^peso) es
astronómicamente inviable para un ataque diferencial real.

### Relación rondas ↔ `intentos_fallidos`

`rondas = min(24, 8 + intentos_fallidos)` es estrictamente creciente hasta saturar en 24
rondas cuando `intentos_fallidos >= 16`. Esto hace que los rangos pedidos no sean igual de
informativos:

- **`intentos_fallidos < 10`** → entre 8 y 17 rondas. Es el único rango que **nunca alcanza**
  las 24 rondas de diseño original: la variante queda permanentemente reducida en rondas
  mientras el contador se mantenga bajo.
- **`intentos_fallidos < 20`** y **`intentos_fallidos < 30`** → ambos permiten llegar a las 24
  rondas completas (saturación en 16); en términos de la fórmula de rondas son
  indistinguibles entre sí.

Aun en el peor caso del primer rango (8 rondas, el mínimo posible), la cota superior
construida ya arroja un peso de varios cientos: el propio diseño de Keccak-f (la
difusión de `theta`+`chi`) impone suficiente margen incluso a 8 rondas frente a
diferenciales clásicos con esta métrica. El riesgo real de que `intentos_fallidos` module el
número de rondas no está tanto en debilitar la resistencia diferencial per se, sino en que
introduce una **superficie de ataque nueva**: el número de rondas (y por tanto el nivel de
seguridad) pasa a depender de un estado mutable potencialmente observable o influenciable por
el atacante (p. ej. forzando intentos fallidos), algo ausente en el Keccak-f original de
rondas fijas.

### `sigma` e `iota` dinámicas: la parte XOR no aporta la resistencia que el README afirma

Como se mostró antes de construir el modelo, tanto el término `state[x][y] ^=
(intentos_fallidos << (x+y))` de `sigma` como el `RC[i] ^ intentos_fallidos` de `iota` son
XORs con una **constante conocida** (no depende de las entradas), por lo que **no tienen
ningún efecto sobre la propagación de diferencias**: para cualquier par de entradas
relacionadas, esa constante se cancela exactamente igual que la constante de ronda del Keccak
original. La afirmación del README ("esto dificultaría toda posibilidad de ataques básicos
que necesitan del determinismo del algoritmo") **no se sostiene frente a un atacante
diferencial clásico**. El único efecto de `sigma` que sí importa es su *rotación*
(equivalente en efecto a un segundo `rho`), y el único efecto real de `intentos_fallidos`
sobre esta métrica de seguridad es el número de rondas y el reordenamiento chi↔pi.

### Conclusión

La propuesta no se rompe frente a criptoanálisis diferencial en ninguno de los escenarios
evaluados (todas las cotas, incluso en el peor caso de 8 rondas, son computacionalmente
inviables de explotar). El punto débil real que este análisis expone no es la resistencia
diferencial de las rondas ejecutadas, sino (a) que el *número* de rondas deja de ser una
constante de diseño y pasa a depender de un contador mutable, reduciendo el margen de
seguridad garantizado por debajo de las 24 rondas estándar mientras `intentos_fallidos < 16`,
y (b) que las partes "dinámicas" que el README presenta como una mejora de seguridad
(`sigma`/`iota` vía XOR de constante) son, frente a un ataque diferencial, criptográficamente
transparentes.


In [12]:
# Tabla resumen: peso -> probabilidad diferencial 2^-peso -> pares estimados necesarios (~2^peso)
def fmt_pairs(weight):
    if weight <= 60:
        return f"~2^{weight:.0f} (~{2**int(weight):.3e})"
    return f"~2^{weight:.0f} (astronomico, sin significado practico)"


print(f"{'z':>3} {'rondas':>6} {'intentos':>8} {'orden':>10} {'peso':>8} {'tipo':>18}  pares estimados")
print("-" * 90)
for r in round1_results:
    orden = "clasico" if r["intentos_fallidos"] < 3 else "variante"
    print(f"{r['z']:>3} {r['rounds']:>6} {r['intentos_fallidos']:>8} {orden:>10} {r['weight']:>8.0f} {'MILP '+r['status']:>18}  {fmt_pairs(r['weight'])}")
for r in heuristic_results:
    orden = "clasico" if r["intentos_fallidos"] < 3 else "variante"
    print(f"{r['z']:>3} {r['rounds']:>6} {r['intentos_fallidos']:>8} {orden:>10} {r['weight_upper_bound']:>8.0f} {'cota superior':>18}  {fmt_pairs(r['weight_upper_bound'])}")


  z rondas intentos      orden     peso               tipo  pares estimados
------------------------------------------------------------------------------------------
  4      1        0    clasico        2       MILP Optimal  ~2^2 (~4.000e+00)
  4      1        5   variante        2       MILP Optimal  ~2^2 (~4.000e+00)
  8      1        0    clasico        2       MILP Optimal  ~2^2 (~4.000e+00)
  8      1        5   variante        2       MILP Optimal  ~2^2 (~4.000e+00)
  4      1        0    clasico        2      cota superior  ~2^2 (~4.000e+00)
  4      1        5   variante        2      cota superior  ~2^2 (~4.000e+00)
  4      2        0    clasico       21      cota superior  ~2^21 (~2.097e+06)
  4      2        5   variante       20      cota superior  ~2^20 (~1.049e+06)
  4      3        0    clasico       62      cota superior  ~2^62 (astronomico, sin significado practico)
  4      3        5   variante       70      cota superior  ~2^70 (astronomico, sin significado pract